# Project goal

Get familiar with small GPT training.  
Very strongly inspired by Andrej Karpathy's [youtube video on nanoGPT](https://youtu.be/kCc8FmEb1nY).

## Load data

In [9]:
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

In [10]:
print("number of characters:", len(text))

number of characters: 1115394


In [11]:
print(text[:500])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor


## Tokenize

Our token are just characters.

In [12]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("number of tokens (codebook size):", vocab_size)
print(chars)

number of tokens (codebook size): 65
['\n', ' ', '!', '$', '&', "'", ',', '-', '.', '3', ':', ';', '?', 'A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'J', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z', 'a', 'b', 'c', 'd', 'e', 'f', 'g', 'h', 'i', 'j', 'k', 'l', 'm', 'n', 'o', 'p', 'q', 'r', 's', 't', 'u', 'v', 'w', 'x', 'y', 'z']


Token encoding = token number in the 'chars' sequence.

In [13]:
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

In [14]:
print(encode('hello world'))
print(decode(encode('hello world')))

[46, 43, 50, 50, 53, 1, 61, 53, 56, 50, 42]
hello world


Alternative: use tiktoken (OpenAI's tokenization library, based on BPE).

In [15]:
import tiktoken
enc = tiktoken.get_encoding("gpt2")
assert enc.decode(enc.encode('hello world')) == 'hello world'

In [16]:
print("vocabulary length (codebook size):",enc.n_vocab)
print(enc.encode('hello world'))

vocabulary length (codebook size): 50257
[31373, 995]


For now, we'll keep the character-level encoding to keep things simple.

## Encode

In [17]:
import torch
data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:500])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

## Split dataset

In [18]:
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

## Extract a chunk from the data

In [19]:
block_size = 8
offset = 0
train_data[offset:offset+block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

This holds 'block_size+1' training examples:
- predict 2nd token based on 1st
- predict 3rd token based on 1st and 2nd
- etc. until predict 'block_size+1' token based on 'block_size' previous ones

Actually, this could also hold even more examples: 'predict 4th token based on 3rd only' works as well. But that would yield an imbalance in the distribution of training context lengths: with the above we have a uniform distribution across context lengths, whereas if we were listing all possible subsequences we would have a large imbalance towards short contexts.

Let's display these 'block_size+1' examples:

In [20]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    context = x[:t+1]
    target = y[t]
    print(f"predict {target} based on {context}")

predict 47 based on tensor([18])
predict 56 based on tensor([18, 47])
predict 57 based on tensor([18, 47, 56])
predict 58 based on tensor([18, 47, 56, 57])
predict 1 based on tensor([18, 47, 56, 57, 58])
predict 15 based on tensor([18, 47, 56, 57, 58,  1])
predict 47 based on tensor([18, 47, 56, 57, 58,  1, 15])
predict 58 based on tensor([18, 47, 56, 57, 58,  1, 15, 47])


## Draw a minibatch

In [21]:
torch.manual_seed(1337)
batch_size = 4 # number of sequences in a minibatch
block_size = 8 # context length

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x,y

In [22]:
xb, yb = get_batch('train')
print('inputs shape:', xb.shape)
print(xb)
print('targets shape:', yb.shape)
print(yb)

inputs shape: torch.Size([4, 8])
tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
targets shape: torch.Size([4, 8])
tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])


In [23]:
for b in range(batch_size):
    for t in range(block_size):
        context = xb[b,:t+1]
        target = yb[b,t]
        print(f"predict {target} based on {context}")

predict 43 based on tensor([24])
predict 58 based on tensor([24, 43])
predict 5 based on tensor([24, 43, 58])
predict 57 based on tensor([24, 43, 58,  5])
predict 1 based on tensor([24, 43, 58,  5, 57])
predict 46 based on tensor([24, 43, 58,  5, 57,  1])
predict 43 based on tensor([24, 43, 58,  5, 57,  1, 46])
predict 39 based on tensor([24, 43, 58,  5, 57,  1, 46, 43])
predict 53 based on tensor([44])
predict 56 based on tensor([44, 53])
predict 1 based on tensor([44, 53, 56])
predict 58 based on tensor([44, 53, 56,  1])
predict 46 based on tensor([44, 53, 56,  1, 58])
predict 39 based on tensor([44, 53, 56,  1, 58, 46])
predict 58 based on tensor([44, 53, 56,  1, 58, 46, 39])
predict 1 based on tensor([44, 53, 56,  1, 58, 46, 39, 58])
predict 58 based on tensor([52])
predict 1 based on tensor([52, 58])
predict 58 based on tensor([52, 58,  1])
predict 46 based on tensor([52, 58,  1, 58])
predict 39 based on tensor([52, 58,  1, 58, 46])
predict 58 based on tensor([52, 58,  1, 58, 46, 

## The bigram language model

The distribution on the next token only depends on the current token. So we have to store a `vocab_size` vector of values for each token, and we have to do that for all tokens. So we have to store a `vocab_size`x`vocab_size` matrix.

In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T)-shaped tensors of integers
        logits = self.token_embedding_table(idx)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

In [25]:
m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(xb.shape)
print(yb.shape)
print(logits.shape)
print(loss)

torch.Size([4, 8])
torch.Size([4, 8])
torch.Size([32, 65])
tensor(4.8786, grad_fn=<NllLossBackward0>)


Vocabulary holds 65 tokens and the model is untrained to should be close to a uniform distribution. So the loss should be close to -ln(1/65):

In [26]:
import numpy as np
print(np.log(vocab_size))

4.174387269895637


Let's add a method that does token generation for `max_new_token` tokens and appends them to the input.

In [27]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, idx, targets=None):
        # idx and targets are both (B,T)-shaped tensors of integers
        logits = self.token_embedding_table(idx)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction for last column of idx
            logits, loss = self(idx)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

Let's generate text starting from token 0.  
So we create a batch with a single sequence, containing a single element, which is the token 0.  
Let's pass it to our 'decode' function, then generate the 100 next tokens.

In [28]:
# first token
idx = torch.zeros((1,1), dtype=torch.long)
decode(idx[0].tolist())

'\n'

In [29]:
m = BigramLanguageModel(vocab_size)
new_seq = m.generate(idx, max_new_tokens=100)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))

predicted sequence:
 tensor([[ 0, 31, 23, 21, 41, 24, 32, 11, 13, 41, 17, 24, 25, 53, 32, 40, 60, 38,
         60,  1, 15, 12, 52, 55,  7, 29, 17,  9,  9, 10, 15, 22, 55, 49, 27, 23,
         20,  7, 55, 11, 10, 50, 39,  2, 53, 47, 63, 61, 49, 20, 48, 45, 15, 46,
         64, 40, 29, 12, 59,  2,  9, 40, 24, 21, 45, 61, 43, 60, 51, 63, 18, 22,
         19, 33, 19, 54,  0, 61, 52, 37, 35, 51, 52, 62, 23, 35, 35, 43, 60,  7,
         58, 16, 55, 36, 17, 56, 34, 23, 24, 45, 22]])
... as a string of characters:
 
SKIcLT;AcELMoTbvZv C?nq-QE33:CJqkOKH-q;:la!oiywkHjgChzbQ?u!3bLIgwevmyFJGUGp
wnYWmnxKWWev-tDqXErVKLgJ


## Training the bigram model

In [30]:
optimizer = torch.optim.Adam(m.parameters(), lr=1e-3) # usual value of 3e-4 is not necessary here as our model is quite small

In [31]:
batch_size = 32
for steps in range(10000):
    xb, yb = get_batch('train')
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
print(loss.item())

2.3796486854553223


Let's try to make a prediction.

In [32]:
new_seq = m.generate(idx, max_new_tokens=400)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))

predicted sequence:
 tensor([[ 0, 50, 50, 53,  1, 40, 56,  8,  1, 39, 60, 43,  1, 39, 60, 47, 39, 57,
         59, 56, 44,  1, 51, 63,  6,  1, 51, 39, 63, 53,  1, 58,  1, 47, 60, 43,
         43,  1, 47, 59, 43, 42, 56, 42,  1, 61, 46, 39, 56,  1, 49, 57, 58, 46,
          1, 63,  1, 46,  1, 40, 53, 56, 39,  1, 57,  1, 40, 43,  1, 46, 43, 57,
         43,  6,  1, 61, 53, 61, 43, 43, 43, 11,  1, 58, 46, 43,  2,  1, 23, 21,
          1,  5, 42, 43,  6,  1, 59, 50, 57, 43, 43, 41, 46, 43, 56, 42,  1, 42,
          1, 53,  1, 40, 50, 50, 50, 39, 52, 42, 53, 11,  0,  0, 35, 46, 43,  6,
          1, 53, 56, 39, 47, 52, 45, 53, 44, 53, 44,  1, 61, 47, 52,  2,  0, 30,
         21, 44, 39, 52, 57,  1, 54, 47, 41, 57, 54, 43, 57, 43, 56, 43, 56,  1,
         46, 43, 43,  1, 58, 46, 39,  6,  0, 32, 27, 18, 53, 52, 49, 12,  1, 51,
         43,  1, 39, 47, 52,  1, 41, 49, 52, 58, 53, 58, 63,  1, 42, 43, 42, 53,
          1, 40, 53,  5, 50, 50, 50, 50,  1, 57, 58,  1, 58, 39,  1, 42, 10,  0,
       

## Wrapping up on data loading and bigram model training

Let's encapsulate all this in a single script.

In [ ]:
# %load bigram.py

This could bet better but it's just a bigram model: only the last character conditions the prediction. So, at this stage, we'd like the model to become a bit more clever and start looking at all tokens in the sequence (not just the last) to make a prediction.

## Introducing token embeddings in the bigram model code

- remove `vocab_size` from constructor since it's a global variable
- introduce `n_embd=32` the token embedding dimension as a global variable
- `token_embedding_table` shape becomes (vocab_size,n_embd)
- calling `token_embedding_table(idx)` now returns the embedding of the token, not the logits for the next token (rename the output from `logits` to `tok_emb`)
- introduce a `lm_head` linear layer that turns an embedding into logits

Introducing this embedding level serves no purpose for now, since the embedding is a lookup table and the lm_head is a linear operation. So, instead of storing `vocab_size=65` probability values for each token line, we now store `n_embd=32` values, and then also store a matrix `lm_head` of shape `(n_embd, vocab_size)` that maps these values back to probability values. This is just a way to introduce token embeddings.

- introduce a `position_embedding_table` embedding that maps positions in the sequence to embeddings
- redefine the forward pass to sum the token embedding with the position embedding and pass the result through `lm_head`

Warning: this causes an indexing problem in the forward pass, as the sequence length can grow larger than `block_size` while the `position_embedding_table` cannot accomodate positions larger than `block_size`. To prevent this, in `generate`, only call `forward` on the last `block_size` elements.

In [40]:
# %load bigram_v2.py

## Personal note

We want to go from $P(s_{t+1}|s_{t})$ to $P(s_{t+1}|s_0,\ldots,s_t)$.  
Naive Bayes learned that through independance assumptions.  
Gaussian Processes learned that by making the Gaussian assumption and learning the kernel parameters.  
Attention will do so with what? Learn the embedding?

## The math trick in self-attention

Currently, decision for the next token relies only on the last one.  
Let's make this dependent on the *average* of all previous ones in the `block_size` sequence. So let's start by computing this average.

In [44]:
import torch

torch.manual_seed(1337)
B,T,C = 4,8,2 # batch, time (context length), channels (codebook size)
x = torch.randn(B,T,C)
x.shape

torch.Size([4, 8, 2])

In [46]:
# we want x[b,t] = mean_{i<=t} x[b,i]
xbow = torch.zeros((B,T,C)) # "bow" is for Bag of Words
for b in range(B):
    for t in range(T):
        xprev = x[b,:t+1] # this has shape (t,C)
        xbow[b,t] = torch.mean(xprev, 0)

In [47]:
x[0]

tensor([[ 0.1808, -0.0700],
        [-0.3596, -0.9152],
        [ 0.6258,  0.0255],
        [ 0.9545,  0.0643],
        [ 0.3612,  1.1679],
        [-1.3499, -0.5102],
        [ 0.2360, -0.2398],
        [-0.9211,  1.5433]])

In [48]:
xbow[0]

tensor([[ 0.1808, -0.0700],
        [-0.0894, -0.4926],
        [ 0.1490, -0.3199],
        [ 0.3504, -0.2238],
        [ 0.3525,  0.0545],
        [ 0.0688, -0.0396],
        [ 0.0927, -0.0682],
        [-0.0341,  0.1332]])

In [49]:
# Version 2: matrix multiplication
# this had complexity BxT
# let's do that more efficiently with matrix multiplication to take advantage of parallelization
wei = torch.tril(torch.ones(T,T)) # torch.tril takes the lower triangular values
wei = wei / wei.sum(1, keepdim=True) # normalize each row
xbow2 = wei @ x 
# when torch sees a matmul (T,T) @ (B,T,C), it morphs it into (B,T,T) @ (B,T,C) 
# and does the matmul element-wise on the first dimension and stacks them. 
# So it does B times a (T,T) @ (T,C) operation.
# So we get a (B,T,C) tensor at the end
print(torch.allclose(xbow,xbow2)) # might fail because of precision issues
print(torch.max(xbow-xbow2))

False
tensor(3.2363e-08)


In [50]:
xbow[0], xbow2[0]

(tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]),
 tensor([[ 0.1808, -0.0700],
         [-0.0894, -0.4926],
         [ 0.1490, -0.3199],
         [ 0.3504, -0.2238],
         [ 0.3525,  0.0545],
         [ 0.0688, -0.0396],
         [ 0.0927, -0.0682],
         [-0.0341,  0.1332]]))

In [51]:
# Version 3: Softmax
tril = torch.tril(torch.ones(T,T))
wei = torch.zeros((T,T))
wei = wei.masked_fill(tril == 0, float('-inf')) # replace 0 values by -inf, in wei, where tril==0
print(wei)
wei = F.softmax(wei, dim=-1) # get the same weight matrix as before
xbow3 = wei @ x
torch.allclose(xbow,xbow3)
print(torch.max(xbow-xbow2))

tensor([[0., -inf, -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., -inf, -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., -inf, -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., -inf, -inf, -inf, -inf],
        [0., 0., 0., 0., 0., -inf, -inf, -inf],
        [0., 0., 0., 0., 0., 0., -inf, -inf],
        [0., 0., 0., 0., 0., 0., 0., -inf],
        [0., 0., 0., 0., 0., 0., 0., 0.]])
tensor(3.2363e-08)


The printed version of `wei` above can be interpreted as *affinities*.  
Actual weights in the matrix that performs the averaging operation, are the softmax over affinities.  
When we predict the nth token, only the previous ones participate and have affinities with the token, not the ones in the future.

## Self-attention

So far, the weights attending to previous positions were uniform (the affinities were all zero). We want these weights to be data dependent.  

For instance, token number 3 might be asking the question "which other tokens should I have affinity with?". This is a *query*. This *query* is directed at all other tokens.  
The query is a vector that stands for "what am I looking for?".

All other tokens will have a *key*. The key is a vector that encodes "what do I contain?" or "what do I mean?".

If the *key* matches the *query*, then the affinity should be high.  

So each token should emit 2 embedding vectors: one query $q(x)$ and one key $k(x)$, based on its embedding $x$. The attention score between a token and any other one is a function of the pair $(q(x),k(x))$.

In dot-product self-attention, queries and keys lie in the same vector space and the affinity is $q(x)^Tk(x)$. The attention score is the softmax of affinities.

Recall that the embedding $x$, as we have defined it so far, holds the information of "which word am I in the codebook?" and the information of "what position am I in?".

In [53]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch size, context window, channels
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) ---> (B,T,T)
# when doing the transpose operation, we want to keep the batch dimension untouched

# we still want to mask future tokens in the attention mechanism
tril = torch.tril(torch.ones(T,T)) 
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ x

print(out.shape)

torch.Size([4, 8, 32])


In [55]:
wei[0]

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.1574, 0.8426, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2088, 0.1646, 0.6266, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5792, 0.1187, 0.1889, 0.1131, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0294, 0.1052, 0.0469, 0.0276, 0.7909, 0.0000, 0.0000, 0.0000],
        [0.0176, 0.2689, 0.0215, 0.0089, 0.6812, 0.0019, 0.0000, 0.0000],
        [0.1691, 0.4066, 0.0438, 0.0416, 0.1048, 0.2012, 0.0329, 0.0000],
        [0.0210, 0.0843, 0.0555, 0.2297, 0.0573, 0.0709, 0.2423, 0.2391]],
       grad_fn=<SelectBackward0>)

In the code above, the prediction for a given token is the average of all its predecessors' embeddings, with weights given by the attention mechanism.  

Let's add one more level of flexibility: we want the prediction for a token to be the average of all its predecessors' *values* $v(x)$ instead of their embeddings $x$.  
Such *values* are "what I will communicate in the answer to the query".

Let's make the value live in a `head_size` dimensional space too.

In [56]:
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)
B,T,C = 4,8,32 # batch size, context window, channels
x = torch.randn(B,T,C)

head_size = 16
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
k = key(x) # (B,T,16)
q = query(x) # (B,T,16)
v = value(x) # (B,T,16)

wei = q @ k.transpose(-2,-1) # (B,T,16) @ (B,16,T) ---> (B,T,T)
# when doing the transpose operation, we want to keep the batch dimension untouched

# we still want to mask future tokens in the attention mechanism
tril = torch.tril(torch.ones(T,T)) 
wei = wei.masked_fill(tril==0, float('-inf'))
wei = F.softmax(wei,dim=-1)
out = wei @ v

print(out.shape)

torch.Size([4, 8, 16])


Notes:
- Attention is a data-dependent communication mechanism across items. It could be applied over other data than time series.
- Positional embedding break the position invariance property to enrich representation (could be regained by removing positional encodings).
- cross-attention refers to the case where the query comes from an element $x$ and the keys and values come from a separate collection of items $y$ living in another space, that we want to pool information from. So we would have $q(x),k(y),v(y)$ elements in the attention mechanism. Self-attention is when $x$ and $y$ live in the same space.
- The operation of computing all $q(x)$, $k(x)$, and $v(x)$, and outputing the next token prediction is called an attention head.
- Scaled dot-product attention:
$$Attention(Q,K,V) = softmax\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$
The scaling is there to ensure unit-variance in the $qk^T$ for unit variance in the inputs $q$ and $k$, as shown in the example below. This is important to keep the inputs of the softmax in a reasonable range, so that its outputs are not too close to zero or one.

In [60]:
k = torch.randn(B,T,head_size)
q = torch.randn(B,T,head_size)
wei = q @ k.transpose(-2,-1)

print(k.var())
print(q.var())
print(wei.var())
print(head_size)

wei = wei * head_size**-.5

print(wei.var())

tensor(1.0104)
tensor(1.0204)
tensor(17.6841)
16
tensor(1.1053)


## Single head attention-based language model

In [79]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,T:]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

Let's use this attention head in our previous code to make the next-token prediction depend on the output of the attention mechanism. Use `head_size`=`n_embd`.

In [80]:
# Single attention head language model
class SingleAttentionHeadLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd) # sa_head stands for self-attention head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_head(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction for last column of idx
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

The script below trains a the model above.

The only changes to the bigram model hyperparameters are:
- learning rate decreased to $1e-3$
- `max_iters` increased to 5000
- `eval_interval` increased to 500

In [84]:
# %load single_attention_head.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

# Single attention head language model
class SingleAttentionHeadLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_head = Head(n_embd) # sa_head stands for self-attention head
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_head(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = SingleAttentionHeadLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.2000, val loss 4.2047
step 500: train loss 2.6902, val loss 2.7078
step 1000: train loss 2.5190, val loss 2.5299
step 1500: train loss 2.4771, val loss 2.4826
step 2000: train loss 2.4404, val loss 2.4521
step 2500: train loss 2.4267, val loss 2.4433
step 3000: train loss 2.4123, val loss 2.4324
step 3500: train loss 2.3948, val loss 2.4210
step 4000: train loss 2.4034, val loss 2.3989
step 4500: train loss 2.3971, val loss 2.4079
predicted sequence:
 tensor([[ 0, 35, 46, 43, 52, 58,  1, 47, 49, 52, 58,  6,  0, 32, 46, 53, 61, 47,
          6,  1, 46, 58,  1, 57, 53, 52,  6,  1, 40, 58, 46,  0,  0, 20, 47, 57,
         43, 58,  1, 40, 53, 40, 43,  1, 39, 50, 43,  8,  0, 31, 10,  0, 27,  7,
          5,  1, 57, 58,  1, 42, 39, 50, 47, 50, 39, 52, 57, 57, 10,  0, 35, 39,
         52, 58,  1, 46, 43,  1, 59, 57,  1, 46, 43,  6,  1, 60, 43, 58, 12,  0,
         35, 43, 42, 47, 50, 39, 57,  1, 39, 58, 43,  1, 39, 61, 47, 41, 43,  1,
         51, 63,  8,  0,  0, 20, 16, 

## Multi-head attention

Instead of a single head, let's use several and just concatenate the output.  
The key idea is to let one head look for a certain type of information (a certain type of query / key / value), while another searches for another type of info, and so on.  
For instance, the first head could look for nearby vowels, the second could look for the presence of the past tense, etc. And then all these independent predictions form the output of the attention layer.

In [86]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

To keep the output dimension constant, use 4 heads, each with an embedding dimension of n_embd/4.

In [88]:
# %load multi_head_attention.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd//4) # 4 heads, each with size n_embd//4
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_heads(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.2227, val loss 4.2226
step 500: train loss 2.6587, val loss 2.6728
step 1000: train loss 2.4971, val loss 2.5057
step 1500: train loss 2.4278, val loss 2.4337
step 2000: train loss 2.3698, val loss 2.3832
step 2500: train loss 2.3399, val loss 2.3549
step 3000: train loss 2.3129, val loss 2.3336
step 3500: train loss 2.2897, val loss 2.3160
step 4000: train loss 2.2877, val loss 2.2859
step 4500: train loss 2.2728, val loss 2.2849
predicted sequence:
 tensor([[ 0, 35, 46, 43, 52, 58,  1, 47, 44,  1, 40, 56, 47, 42, 41, 53, 61, 42,
          6,  1, 61, 46, 47, 57,  1, 40, 63, 43, 56,  1, 58, 46, 39, 58,  1, 57,
         43, 58,  1, 40, 53, 40, 43,  1, 58, 53, 43,  1, 39, 52, 58, 46, 56,  7,
         39, 52, 42,  1, 51, 43, 39, 50, 50, 43, 39, 59, 57, 57, 10,  0, 35, 39,
         56, 58, 46,  1, 44, 53, 59, 50, 55, 59, 43,  6,  1, 60, 43, 58, 12,  0,
         35, 43, 42, 58, 50, 39, 63,  1, 39, 52, 43, 57,  1, 61, 47, 41, 43,  1,
         51, 63,  8,  0,  0, 20, 16, 

## Adding a feedforward layer after the attention output

Currently, the output of all attention heads is just concatenated and fed to the (linear) language model head to compute the logits. Let's add an inbetween feedforward ReLU layer so that there can be some "reasoning" on the output of the attention heads.

In [89]:
class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

In [93]:
# %load multi_head_attention_v2.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self,x):
        return torch.cat([h(x) for h in self.heads], dim=-1)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU()
        )

    def forward(self, x):
        return self.net(x)

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.sa_heads = MultiHeadAttention(num_heads=4, head_size=n_embd//4) # 4 heads, each with size n_embd//4
        self.ffwd = FeedForward(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.sa_heads(x)
        x = self.ffwd(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.1996, val loss 4.1995
step 500: train loss 2.5990, val loss 2.6072
step 1000: train loss 2.4622, val loss 2.4642
step 1500: train loss 2.3964, val loss 2.3937
step 2000: train loss 2.3277, val loss 2.3451
step 2500: train loss 2.2993, val loss 2.3193
step 3000: train loss 2.2801, val loss 2.2907
step 3500: train loss 2.2478, val loss 2.2708
step 4000: train loss 2.2413, val loss 2.2451
step 4500: train loss 2.2262, val loss 2.2394
predicted sequence:
 tensor([[ 0, 13, 52, 42,  1, 58, 46, 43,  1, 30, 53, 56,  0, 32, 46, 53, 61,  1,
         39, 52, 42,  1, 47, 57,  1, 58, 53,  1, 40, 50, 43,  0, 40, 39, 47, 56,
         43, 58,  1, 40, 59, 40, 43,  1, 58, 53,  1, 58, 39, 56, 60, 43, 56,  7,
          5,  1, 51, 63,  1, 42, 39, 50, 50,  1, 39, 52, 42, 57,  1, 39, 56,  1,
         46, 47, 54, 46, 43,  1, 59, 57,  1, 46, 39, 58,  1, 58, 53, 58, 12,  0,
         35, 43, 42, 58, 50, 39, 42,  1, 39, 52, 43,  1, 39, 61,  1, 41, 56, 59,
         54,  1, 39, 52, 42,  1, 52, 

## Blocks of multihead attention and reasoning, with skip connections

In [95]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = self.sa(x)
        x = self.ffwd(x)
        return x

Let's build a network with a feedforward sequence of 3 such blocks.

The network is starting to become quite deep and optimization would start to struggle. So let's also add skip connections to facilitate gradient flow.

In [96]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(x) # skip connection
        x = x + self.ffwd(x) # skip connection
        return x

Since we have added the skip connections, we are re-injecting the result of computation (`sa(x)` and `ffwd(x)`) into the "residual pathway" (`x`). So we need to introduce an degree of freedom to project these results onto that pathway: we need to add a linear layer at the end of the `MultiHeadAttention` and `Feedforward` classes.

In [97]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.prof(out)

In [98]:
class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, n_embd),
            nn.ReLU(),
            nn.Linear(n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

One last improvement coming from the *Attention is all you need* paper: the inner layer in the `FeedForward` module has dimensionality `4*n_embd`.

In [102]:
# %load multi_head_attention_v3.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)

    def forward(self, x):
        x = x + self.sa(x) # skip connection
        x = x + self.ffwd(x) # skip connection
        return x

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.blocks(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.6255, val loss 4.6233
step 500: train loss 2.3874, val loss 2.3842
step 1000: train loss 2.2689, val loss 2.2678
step 1500: train loss 2.1878, val loss 2.2103
step 2000: train loss 2.1452, val loss 2.1820
step 2500: train loss 2.1045, val loss 2.1519
step 3000: train loss 2.0677, val loss 2.1426
step 3500: train loss 2.0576, val loss 2.1176
step 4000: train loss 2.0248, val loss 2.1081
step 4500: train loss 2.0023, val loss 2.1032
predicted sequence:
 tensor([[ 0, 13, 52, 42,  1, 58, 46, 43, 63,  1, 40, 56, 47, 42, 41, 43, 12,  0,
          0, 31, 27, 30, 27, 30, 16,  1, 17, 42, 39, 58, 43,  0, 40, 39, 41, 43,
         43, 42,  1, 40, 59, 40, 43,  1, 39,  1, 43, 52, 39, 51, 43, 45, 56, 39,
         60, 43, 57,  1, 51, 43, 12,  0, 24, 43, 39, 56, 57, 43, 10,  0, 35, 39,
         41, 58,  1, 46, 43,  1, 59, 57,  1, 46, 43, 39, 56, 42,  8,  1, 35, 39,
         56, 42,  1, 58, 46, 39, 58,  1, 39, 52, 43,  1, 39, 61, 39, 63,  6,  1,
         51, 63,  1, 44, 43, 39, 52, 

## Layer normalization

Let's make the optimization more efficient by implementing layer normalization.

In [103]:
class LayerNorm:
    def __init__(self, dim, eps=1e-5):
        self.eps = eps
        self.gamma = torch.ones(dim)
        self.beta = torch.zeros(dim)

    def __call__(self, x):
        xmean = x.mean(1, keepdim=True) # Layer-wise mean
        xvar = x.var(1, keepdim=True) # Layer-wise variance
        xhat = (x-xmean)/torch.sqrt(xvar + self.eps) # normalize to unit variance
        out = self.gamma * xhat + self.beta
        return out

    def parameters(self):
        return [self.gamma, self.beta]

torch.manual_seed(1337)
module = LayerNorm(100)
x = torch.randn(32, 100) # batch of 32 100-dimensional vectors
x = module(x)

print(x.shape)
print(x[:,0].mean(), x[:,0].std())
print(x[0,:].mean(), x[0,:].std())

torch.Size([32, 100])
tensor(0.1469) tensor(0.8803)
tensor(-9.5367e-09) tensor(1.0000)


Slight deviation from the original *Attention is all you need paper*: recent years have shown that implementing layer normalization *before* multi-head attention and feedforward modules was beneficial compared to putting them after. So that's what we will do.

Also, we'll use pytorch built-in `LayerNorm` layer.

In [104]:
class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip connection
        x = x + self.ffwd(self.ln2(x)) # skip connection
        return x

Also add layer normalization after the 3 `Block`, so we feed a clean, normalized vector into the final logits prediction output layer.

In [108]:
# %load multi_head_attention_v4.py
import torch
import torch.nn as nn
from torch.nn import functional as F

# hyperparameters
batch_size = 32
block_size = 8
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 200
n_embd = 32

# seeding (for reproducibility)
torch.manual_seed(1337)

# open text file
filename = 'tinyshakespeare.txt' # 'tinymoliere.txt'
with open(filename, 'r', encoding='utf-8') as f:
    text = f.read()

# tokens
chars = sorted(list(set(text)))
vocab_size = len(chars)

# encode and decode strings
stoi = { ch:i for i,ch in enumerate(chars) }
itos = { i:ch for i,ch in enumerate(chars) }
encode = lambda s: [stoi[c] for c in s] # encode a string as a list of integers
decode = lambda l: ''.join([itos[i] for i in l]) # decode a list of integers as a string

# split dataset
data = torch.tensor(encode(text), dtype=torch.long)
n = int(.9*len(data))
train_data = data[:n]
val_data = data[n:]

# data loading
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    x, y = x.to(device), y.to(device)
    return x,y

# estimate training and validation losses (for monitoring purposes)
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X,Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# single attention head
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        self.register_buffer('tril', torch.tril(torch.ones(block_size,block_size)))

    def forward(self,x):
        B,T,C = x.shape
        k = self.key(x)   # B,T,C warning, here C stands for head_size
        q = self.query(x) # B,T,C
        # compute affinities and attention scores
        wei = q @ k.transpose(-2,-1) * C**-.5 # (B,T,T)
        wei = wei.masked_fill(self.tril[:T,:T]==0, float('-inf')) # (B,T,T)
        wei = F.softmax(wei, dim=-1) # (B,T,T)
        # aggregate values
        v = self.value(x) # (B,T,C)
        out = wei @ v # (B,T,C)
        return out

class MultiHeadAttention(nn.Module):
    """ multiple heads of attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embd, n_embd)

    def forward(self,x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        return self.proj(out)

class FeedForward(nn.Module):
    """ ReLU layer """

    def __init__(self, n_embd):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4*n_embd),
            nn.ReLU(),
            nn.Linear(4*n_embd, n_embd),
        )

    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication then computation """

    def __init__(self, n_embd, n_head):
        super().__init__()
        head_size = n_embd // n_head
        self.sa = MultiHeadAttention(n_head, head_size)
        self.ffwd = FeedForward(n_embd)
        self.ln1 = nn.LayerNorm(n_embd)
        self.ln2 = nn.LayerNorm(n_embd)

    def forward(self, x):
        x = x + self.sa(self.ln1(x)) # skip connection
        x = x + self.ffwd(self.ln2(x)) # skip connection
        return x

# Single attention head language model
class MultiHeadAttentionLanguageModel(nn.Module):
    def __init__(self):
        super().__init__()
        # The logits for the next token are simply stored in a lookup table
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        self.blocks = nn.Sequential(
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            Block(n_embd, n_head=4),
            nn.LayerNorm(n_embd),
        )
        self.lm_head = nn.Linear(n_embd, vocab_size)

    def forward(self, idx, targets=None):
        B, T  = idx.shape
        # idx and targets are both (B,T)-shaped tensors of integers
        tok_emb = self.token_embedding_table(idx) # (B,T,C)
        pos_emb = self.position_embedding_table(torch.arange(T, device=device)) # (T,C) 
        # warning here, T might grow larger than block_size if we generate more tokens than block_size, which will cause an index error in the previous line, but we can ignore that for now.
        x = tok_emb + pos_emb # the sum broadcasts the (T,C) tensor across batches
        x = self.blocks(x)
        logits = self.lm_head(x)
        # at this point, logits are a (B,T,C)-shaped tensor
        # Pytorch expects a (n_data,n_classes) tensor to compute a CE loss
        # so we reshape the logits if we need to compute a loss
        # otherwise, keep the logits in the shape they are
        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T,C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is a (B,T) array of indices
        for _ in range(max_new_tokens):
            # get prediction
            idx_cond = idx[:,-block_size:] # crop idx to the last block_size tokens
            logits, loss = self(idx_cond)
            logits = logits[:,-1,:]
            # apply softmax
            probs = F.softmax(logits, dim=-1)
            # sample next token
            idx_next = torch.multinomial(probs, num_samples=1)
            # append to sequence
            idx = torch.cat((idx,idx_next), dim=1)
        return idx

model = MultiHeadAttentionLanguageModel()
m = model.to(device)

# optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
for iter in range(max_iters):
    # periodically evaluate the validation loss
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss {losses['train']:.4f}, val loss {losses['val']:.4f}")
    # sample minibatch
    xb, yb = get_batch('train')
    # evaluate loss and take gradient step
    logits, loss = m(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    
# generate from model
idx = torch.zeros((1,1), dtype=torch.long, device=device)
new_seq = m.generate(idx, max_new_tokens=500)
print("predicted sequence:\n", new_seq)
print("... as a string of characters:\n", decode(new_seq[0].tolist()))


step 0: train loss 4.3103, val loss 4.3097
step 500: train loss 2.3986, val loss 2.3994
step 1000: train loss 2.2640, val loss 2.2658
step 1500: train loss 2.1660, val loss 2.1890
step 2000: train loss 2.1297, val loss 2.1660
step 2500: train loss 2.0793, val loss 2.1284
step 3000: train loss 2.0499, val loss 2.1242
step 3500: train loss 2.0420, val loss 2.1038
step 4000: train loss 2.0107, val loss 2.0927
step 4500: train loss 1.9912, val loss 2.0968
predicted sequence:
 tensor([[ 0,  0, 37, 27, 30, 35, 13, 30, 21, 34, 10,  0, 26, 53, 41, 46, 61, 47,
         50, 50,  1, 58, 46, 39, 58,  1, 44, 43, 50, 58,  1, 51, 39, 42, 47, 57,
         43, 42,  1, 40, 59, 40, 43, 56,  1, 39, 52, 42,  1, 31, 39, 45, 56, 59,
         39, 52, 57,  1, 51, 43, 39, 50, 50, 43, 42,  0, 25, 63,  1, 39, 56, 58,
         46, 47, 54,  1, 44, 53, 59, 50, 55,  1, 39, 50, 50,  1, 40, 59, 40, 50,
         43,  1, 42, 47, 50, 39, 46, 53, 39, 58, 43, 57,  1, 61, 47, 50, 50,  1,
         51, 63,  1, 44, 43, 39, 56, 

## Scaling up